# 01 · DNS triage on a public capture

**Data:** `dns.cap` from the [Wireshark sample captures](https://wiki.wireshark.org/SampleCaptures), downloaded on demand and verified by sha256 (see `samples.json`).

**Questions an analyst asks first:** who is resolving what, which lookups fail, and is anything unanswered? Failed internal lookups (NXDOMAIN on `_ldap._tcp…_msdcs`) are a classic sign of a host that cannot find its domain controller.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # nbmake runs notebooks from their own folder
import fetch_samples
import pcaplib

In [ ]:
path = fetch_samples.ensure("dns.cap")
msgs = pcaplib.dns_messages(path)
queries = [m for m in msgs if not m["response"]]
responses = [m for m in msgs if m["response"]]
print(f"{len(msgs)} DNS messages: {len(queries)} queries, {len(responses)} responses")
assert len(queries) == len(responses) == 19

## Query types
A mix of record types in one capture usually means someone was testing a resolver by hand (`dig`/`nslookup`), not normal client traffic.

In [ ]:
from collections import Counter

qtypes = Counter(m["qtype"] for m in queries)
for qtype, n in qtypes.most_common():
    print(f"{qtype:>5}  {n}")
assert qtypes["AAAA"] == 6 and qtypes["SRV"] == 3

## Failed lookups (NXDOMAIN)

In [ ]:
failed = pcaplib.nxdomain_names(msgs)
for name in failed:
    print(name)
assert "www.example.notginh" in failed
assert sum(name.endswith(".local") for name in failed) == 4

**Reading:** four of the five NXDOMAIN answers are Active Directory SRV and host lookups for a `.local` domain sent to a resolver that does not serve it. In an incident that points to a mis-configured DNS server on the client (for example a VPN pushing public resolvers), not to an attack.

## Unanswered queries

In [ ]:
lost = pcaplib.unanswered_dns(msgs)
print(f"{len(lost)} unanswered queries")
assert lost == []

Every query got an answer, so there is no sign of filtering or packet loss in this capture.